# 实验一：YOLO 模型准备与 OM 转换

Ascend 端侧推理通常使用 OM 模型。常见流程是先从训练框架导出 ONNX，再使用 ATC 根据目标芯片和输入 shape 转换为 OM。

## 转换流程

```text
PyTorch / 训练框架权重
        ↓ export
ONNX 模型
        ↓ atc --soc_version=Ascend310B4
OM 模型
        ↓ PyACL 加载
Atlas 200I DK A2 / Ascend 310B4 端侧推理
```

转换前最重要的是确认输入名、输入 shape、layout、动态/静态尺寸、目标芯片 `soc_version` 和后处理输出格式。

## 可选模型对比

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">模型</th>
      <th style="text-align: left;">是否推荐</th>
      <th style="text-align: left;">原因</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">YOLOv5s，推荐主线</td>
      <td style="text-align: left;">推荐</td>
      <td style="text-align: left;">与本实验默认路径和输出 shape 匹配，COCO 80 类，640 输入，模型较小，适合教学演示和后处理优化</td>
    </tr>
    <tr>
      <td style="text-align: left;">YOLOv5n，备用轻量模型</td>
      <td style="text-align: left;">可选</td>
      <td style="text-align: left;">参数量更小、推理更轻，适合内存紧张或只想快速验证链路的情况；后处理输出结构与 YOLOv5s 相同</td>
    </tr>
    <tr>
      <td style="text-align: left;">YOLOv5m/l/x</td>
      <td style="text-align: left;">不建议首跑</td>
      <td style="text-align: left;">模型更大，转换和端侧运行更容易受内存、算子支持和耗时影响，不适合作为本实验第一次跑通的模型</td>
    </tr>
    <tr>
      <td style="text-align: left;">YOLOv8/YOLOv10 等新模型</td>
      <td style="text-align: left;">不作为本实验默认</td>
      <td style="text-align: left;">输出结构与后处理代码不同，需要改 decode 和 NMS 逻辑，容易偏离本实验重点</td>
    </tr>
  </tbody>
</table>

## 下载 YOLOv5s 模型并导出 ONNX

如果下载速度较慢或网络连接不稳定，可从先下载模型至本地再通过 JupyterLab 上传。

__重要提醒：导出 ONNX 需要 PyTorch、YOLOv5、onnx、opencv、numpy 等一堆 Python 依赖。开发板虽然有 CANN/PyACL/运行环境，但它不一定完整配置了 PyTorch 训练或导出环境，而且开发板性能也弱，容易卡在依赖、版本、内存上。__


1. 进入本实验目录（根据实际情况自行调整）。

```bash
cd /home/HwHiAiUser/samples/notebooks/04_YOLO_Edge_TBE_Ascend
mkdir -p models 
mkdir -p src/third_party
```

2. 下载 YOLOv5s 权重，作为本实验默认模型。

```bash
wget -c -O models/yolov5s.pt \
  https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
```

3. 获取 YOLOv5 导出脚本

```bash
cd src/third_party
git clone -b v7.0 https://github.com/ultralytics/yolov5.git
cd yolov5
```

4. 安装导出 ONNX 所需依赖。

```bash
python3 -m pip install -r requirements.txt
python3 -m pip install onnx
```

5. 导出静态输入 ONNX。

输出文件会写到 ../../models/yolov5s.onnx；如果这一步报错，可能是所需依赖与模型版本不兼容的问题，参考日志信息重新下载相关依赖即可。另外新版 PyTorch 默认非常严格，torch.load() 不允许直接反序列化 YOLOv5 这种完整模型对象，所以应先执行下面这行命令。

```bash
export TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1

python3 export.py \
  --weights ../../models/yolov5s.pt \
  --include onnx \
  --imgsz 640 \
  --batch-size 1 \
  --opset 12
  
ls -lh ../../models/yolov5s.onnx
```

如果你的环境没有 `wget`，可以使用：

```bash
curl -L -o models/yolov5s.pt \
  https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s.pt
```


In [ ]:
# ====== 1. 读取模型转换配置 ======
from src.scripts.config_utils import load_config

cfg = load_config('src/configs/yolo_edge.yaml')
for key, value in cfg['model'].items():
    print(f'{key:16s}: {value}')

## 导出 ONNX 的注意事项

<table style="margin-left: 0; margin-right: auto; text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">项目</th>
      <th style="text-align: left;">建议</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">opset</td>
      <td style="text-align: left;">与模型仓库和 CANN 支持范围匹配，优先使用稳定版本</td>
    </tr>
    <tr>
      <td style="text-align: left;">输入 shape</td>
      <td style="text-align: left;">端侧部署优先静态 shape，例如 <code>1,3,640,640</code></td>
    </tr>
    <tr>
      <td style="text-align: left;">后处理</td>
      <td style="text-align: left;">可选择保留在模型外，便于替换为自定义 NPU 后处理</td>
    </tr>
    <tr>
      <td style="text-align: left;">NMS</td>
      <td style="text-align: left;">本实验建议模型输出 raw predictions，NMS 在后处理 pipeline 中实现</td>
    </tr>
  </tbody>
</table>

In [ ]:
# ====== 2. 生成 ATC 转换命令 ======
from pathlib import Path
import yaml

config_path = Path('src/configs/yolo_edge.yaml')
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))

model = cfg['model']

cmd = f"""
atc \\
  --framework=5 \\
  --model={model['onnx_path']} \\
  --output={str(model['om_path']).removesuffix('.om')} \\
  --input_shape={model['input_name']}:{','.join(map(str, model['input_shape']))} \\
  --input_format={model['input_format']} \\
  --soc_version={model['soc_version']} \\
  --precision_mode={model['precision_mode']} \\
  --log=info
"""
print(cmd)

## ACT 模型转换

导出 `models/yolov5s.onnx` 后，需要使用 ATC 根据目标芯片和输入 shape 转换为 OM。具体步骤如下：

1. 查找找 CANN 环境脚本。

```bash
find /usr/local/Ascend /home -name set_env.sh 2>/dev/null
```
2. 找到下面这个环境脚本并执行。

```bash
source /usr/local/Ascend/ascend-toolkit/7.0.RC1/aarch64-linux/script/set_env.sh
```
3. 检查 atc。

```bash
which atc
atc --help | head
```
4. 执行转换命令。

```bash
SOC_VERSION=Ascend310B4 OM_PATH=models/yolov5s_310b4 bash src/scripts/export_onnx_to_om.sh
```
5. 看到`ATC run success`说明转换成功，最后确认：

```bash
ls -lh models/yolov5s_310b4.om
```

6. 异常处理。

开发板资源较小，而 ATC 默认会并行编译多个算子，编译 YOLOv5s 时有可能把 TBE 编译进程压崩了。遇到这种情况，首先清掉残留 ATC 进程：（即残留 PID）

```bash
ps -ef | grep "atc\|yolov5s.onnx" | grep -v grep
```
接着重新启动环境脚本。

```bash
source /usr/local/Ascend/ascend-toolkit/7.0.RC1/aarch64-linux/script/set_env.sh
```
降并行后重新执行 ATC。

```bash
export TE_PARALLEL_COMPILER=1
atc --model=models/yolov5s.onnx \
    --framework=5 \
    --output=models/yolov5s_310b4 \
    --soc_version=Ascend310B4 \
    --input_shape="images:1,3,640,640" \
    --input_format=NCHW \
    --log=info
```

脚本默认使用：

```bash
ONNX_PATH=models/yolov5s.onnx
OM_PATH=models/yolov5s_310b4
SOC_VERSION=Ascend310B4
INPUT_SHAPE=images:1,3,640,640
PRECISION_MODE=allow_mix_precision
```

如果转换失败，优先检查：ONNX 输入名是否为 `images`、shape 是否为 `1,3,640,640`、`soc_version` 是否与 `npu-smi info` 中的芯片 Name 匹配、当前 shell 是否已加载 CANN 环境变量。

In [ ]:
# ====== 3. 检查模型文件是否存在 ======
from pathlib import Path
for item in ['onnx_path', 'om_path']:
    path = Path(cfg['model'][item])
    print(f'{item:10s}: {path}  exists={path.exists()}')

print('\n没有模型文件时也可以继续学习后续章节；PyACL 脚本会进入 dry-run 演示流程。')

## 课后练习

请根据本节实验内容完成以下练习。题型包含单选题、多选题、判断题、填空题、简答题和代码设计题。

1. (单选题) 本实验中 ATC 转换的直接目标是什么？
   - A. 把 VOC XML 转成 YOLO txt
   - B. 把 YOLOv5s ONNX 模型转换为 Ascend 310B4 可执行的 OM 模型
   - C. 把 bus.jpg 转成权重文件
   - D. 生成 MindStudio Profiling 报告

2. (单选题) YOLOv5s 模型输入在本实验配置中采用的格式是？
   - A. NHWC
   - B. NCHW
   - C. HWC
   - D. CHWN

3. (单选题) 针对 Atlas 200I DK A2 / Ascend 310B4，ATC 的 `--soc_version` 应设置为？
   - A. Ascend910
   - B. Ascend310B4
   - C. Ascend310P3
   - D. CPU

4. (单选题) `models/yolov5s_310b4.om` 与 `bus.jpg` 的关系最准确的是？
   - A. OM 是由 bus.jpg 训练得到的
   - B. OM 是模型文件，bus.jpg 是后续推理验证输入，二者不是同一种文件
   - C. bus.jpg 会被 ATC 转换成 OM
   - D. OM 只保存 bus.jpg 的检测框

5. (多选题) ATC 转换时通常需要关注哪些参数？
   - A. --model
   - B. --output
   - C. --input_shape
   - D. --soc_version

6. (多选题) 模型转换前应检查哪些内容？
   - A. ONNX 文件是否存在
   - B. CANN 环境变量是否生效
   - C. 输入 shape 与配置是否一致
   - D. 目标芯片型号是否正确

7. (多选题) YOLOv5s OM 输出 shape `[1,25200,85]` 中，85 通常包含哪些信息？
   - A. 4 个边框坐标
   - B. 1 个目标置信度
   - C. 80 个 COCO 类别分数
   - D. NPU 温度

8. (判断题) 只要 ONNX 文件存在，ATC 转换一定成功，不需要 CANN 环境。

9. (判断题) OM 模型是面向 Ascend 设备部署的离线模型格式。

10. (填空题) 本实验配置中的模型输入 shape 是 `____`。

11. (填空题) ATC 转换成功后，输出文件通常以 `____` 作为后缀。

12. (简答题) 为什么 ONNX 到 OM 的转换不能随便换 soc_version？

13. (简答题) 为什么本实验使用 bus.jpg 作为样例输入？

14. (简答题) 转换后如何初步判断 OM 文件是可用于后续实验的？

15. (代码设计题) 写一条 ATC 转换命令，将 `models/yolov5s.onnx` 转为 Ascend310B4 的 OM。

> 参考答案见 answer/04.02_yolo_model_preparation_and_om_conversion_answer.ipynb。
